# 06 · Multimodal output

Image generation and text-to-speech via OpenAI. Each image costs a few cents; speech is cheap.
Both plug into Gradio as `gr.Image` (PIL image) and `gr.Audio` (file path or bytes).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, for `toolkit`

In [ ]:
import base64
from io import BytesIO

from IPython.display import Audio, display
from PIL import Image

from toolkit import get_client

openai = get_client("openai")

In [ ]:
def poster(subject: str) -> Image.Image:
    r = openai.images.generate(
        model="gpt-image-1-mini",
        prompt=f"Minimalist travel poster of {subject}, flat colours, no text",
        size="1024x1024",
    )
    return Image.open(BytesIO(base64.b64decode(r.data[0].b64_json)))


img = poster("Lagos at dusk, Third Mainland Bridge")
img.resize((512, 512))

In [ ]:
speech = openai.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input="Your flight to Nairobi is confirmed. Reference H A one F two E three.",
    instructions="Warm, professional airline agent.",
)
display(Audio(speech.content, autoplay=False))

Notes:
- Generation is slow (5-20 s per image), so run it after the text reply has streamed and render it
  asynchronously in the UI. `travel_desk` does this as the final `yield`.
- Spell out references and codes in TTS input; voices tend to read "HA1F2E3D" as a word.